In [2]:
import pandas as pd
df = pd.read_csv("/home/noah/HALLO_encoder_collection/data_raw/ssl_dclde_dori/20261022_180927/annotations.csv")
df['Source'].unique()

array(['dclde', 'dori', 'onc_background'], dtype=object)

In [3]:
df2 = pd.read_csv("/home/noah/HALLO_encoder_collection/data_raw/DORI/20261022_190549/annotations.csv")
len(df2['Dataset'].unique())

/tmp/ipykernel_1300999/1494261759.py:1: DtypeWarning: Columns (5,7,10,12,13,15,16,23,25,28,32,33,34,35,36,37,38,39,40) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv("/home/noah/HALLO_encoder_collection/data_raw/DORI/20261022_190549/annotations.csv")


35

In [4]:
labelled = df2[df2['Labels'] != 'Background']
print(f"Labelled:   {labelled['Duration'].sum() / 3600:.2f} h ({len(labelled)} annotations)")
print((labelled.groupby('Labels')['Duration'].sum() / 3600).round(2))
print(f"Unlabelled: {df['Duration'].sum() / 3600:.2f} h ({len(df)} windows)")
print((df.groupby('Source')['Duration'].sum() / 3600).round(2))

Labelled:   64.87 h (214655 annotations)
Labels
AB         2.45
HW        22.64
KW_und     1.73
NRKW       1.65
OKW        1.19
SAR        1.93
SRKW      21.61
TKW        9.95
UndBio     1.73
Name: Duration, dtype: float64
Unlabelled: 3872.56 h (4647077 windows)
Source
dclde             1064.68
dori              2795.38
onc_background      12.50
Name: Duration, dtype: float64


In [ ]:
# spectrograms (3 s windows) each A / B / C pretraining set and fine-tune site actually uses: rows in any split,
# labelled = not Background / Unlabelled (AB and UndBio count as labelled)
R = "/home/noah/HALLO_encoder_collection/data_raw/"
D = R + "DORI/20261022_190549/"
sets = {  # name: (annotations, splits)
    "C SSL (all unlabelled)": (R + "ssl_dclde_dori/20261022_180927/annotations.csv", R + "ssl_dclde_dori/20261022_180927/splits.csv"),
    "B SSL (source clips)": (R + "ssl_dori_source/20261026_182504/annotations.csv", R + "ssl_dori_source/20261026_182504/splits.csv"),
    "A supervised (source)": (D + "annotations_bgcap10000.csv", D + "splits_bgcap10000.csv"),
    **{f"FT {s}": (D + f"annotations_bgcap10000_ft_{s}.csv", D + f"splits_pretrain_cmp_ft_{s}.csv")
       for s in ["dori_venus_strait_of_georgia", "limekiln", "tekteksen"]},
}
rows = []
for name, (ann, spl) in sets.items():
    s = pd.read_csv(spl).set_index("uid")
    if name.startswith("A"):
        s = s[s.fold_0 != "test"]  # test = the 4 held-out fine-tune deployments
    labels = pd.read_csv(ann, usecols=["uid", "Labels"], low_memory=False).set_index("uid").loc[s.index[s.notna().any(axis=1)], "Labels"]
    bg = labels.isin(["Background", "Unlabelled"])
    rows.append({"set": name, "spectrograms": len(labels), "labelled": (~bg).sum(), "background / unlabelled": bg.sum()})
pd.DataFrame(rows).set_index("set")

In [ ]:
# files holding at least one labelled (non-Background) annotation, and their full recording length in hours
# (file length = the sum of C's back-to-back 3 s tiles of that file)
file_h = df.groupby("LocalPath")["Duration"].sum() / 3600
lab_files = {"all DORI annotations": df2.loc[df2["Labels"] != "Background", "LocalPath"].dropna().unique()}
for name, (ann, spl) in list(sets.items())[2:]:  # A and the fine-tunes (B = A's clips, C has no labels)
    a = pd.read_csv(ann, usecols=["uid", "Labels", "LocalPath"], low_memory=False).merge(pd.read_csv(spl), on="uid")
    if name.startswith("A"):
        a = a[a.fold_0 != "test"]
    lab_files[name] = a.loc[a["Labels"] != "Background", "LocalPath"].unique()
pd.DataFrame({name: {"files": len(f), "recording hours": file_h.reindex(f).sum().round(1)} for name, f in lab_files.items()}).T.astype({"files": int})

In [ ]:
# one SRKW call (Tekteksen, call type S04) as the model sees it: 3 s window centred on the call, 00_base.yaml's mel settings
import librosa, librosa.display, matplotlib.pyplot as plt
ft = pd.read_csv(D + "annotations_bgcap10000_ft_tekteksen.csv", low_memory=False)
call = ft[(ft.Labels == "SRKW") & (ft.CallType == "S04")].iloc[0]
y, sr = librosa.load(call.LocalPath, sr=24000, offset=call.CenterTime - 1.5, duration=3.0)
S = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_fft=1024, hop_length=128, n_mels=256, fmin=200, fmax=12000))
fig, ax = plt.subplots(figsize=(10, 4))
img = librosa.display.specshow(S, sr=sr, hop_length=128, x_axis="time", y_axis="mel", fmin=200, fmax=12000, ax=ax)
ax.axvspan(call.FileBeginSec - (call.CenterTime - 1.5), call.FileEndSec - (call.CenterTime - 1.5), color="w", alpha=0.15)
ax.set_title(f"SRKW {call.CallType}: {call.LocalPath.split('/')[-1]} @ {call.FileBeginSec:.1f} s")
fig.colorbar(img, ax=ax, format="%+.0f dB")
plt.show()